# Clustering methods

How periods are **grouped**, set with `ClusterConfig(method=...)`. (What each group *becomes* is a
separate lever — see [Representations](representations.ipynb).)

| method | how it groups | notes |
|--------|---------------|-------|
| `hierarchical` | merges the closest periods step by step | the **default** — robust, deterministic, fast |
| `kmeans` | minimises distance to cluster centroids | often the most accurate |
| `kmedoids` | like k-means but centres are **real** periods | exact (MILP), [very slow at scale](runtime.ipynb) |
| `kmaxoids` | centres pushed toward extremes | preserves spread, not the average |
| `averaging` | splits the timeline into equal chunks | simplest baseline, ignores similarity |
| `contiguous` | only merges temporally adjacent periods | keeps calendar order |

In [ ]:
import pandas as pd
import plotly.io as pio

import tsam

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("../data/testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]  # six weeks of hourly data

## Compare them at the same size

In [ ]:
from tsam import ClusterConfig

methods = ["hierarchical", "kmeans", "kmedoids", "kmaxoids", "averaging", "contiguous"]
rows = {}
for method in methods:
    r = tsam.aggregate(
        data, n_clusters=6, period_duration="1D", cluster=ClusterConfig(method=method)
    )
    rows[method] = {"mean RMSE": round(float(r.accuracy.rmse.mean()), 4)}
pd.DataFrame(rows).T.sort_values("mean RMSE")

Runtime differs far more than accuracy does, and the ranking changes with dataset size — see
[How long will this take?](runtime.ipynb).

## The same week, four ways

In [ ]:
import plotly.express as px

week = slice("2010-01-11", "2010-01-17")
frames = [
    pd.DataFrame(
        {
            "time": data.loc[week].index,
            "Load": data.loc[week, "Load"].values,
            "method": "original",
        }
    )
]
for method in ["hierarchical", "kmeans", "kmedoids", "averaging"]:
    r = tsam.aggregate(
        data, n_clusters=6, period_duration="1D", cluster=ClusterConfig(method=method)
    )
    s = r.reconstructed.loc[week, "Load"]
    frames.append(pd.DataFrame({"time": s.index, "Load": s.values, "method": method}))
px.line(
    pd.concat(frames),
    x="time",
    y="Load",
    color="method",
    title="One week reconstructed, by clustering method",
)

## Cluster ids carry no chronology

A typical period stands in for days scattered all over the calendar. `result.cluster_representatives`
is a **menu**, not a compressed timeline — the sequence lives in `result.cluster_assignments`.

Only `contiguous` and `averaging` group days into contiguous calendar runs. Even they do **not**
number their clusters in calendar order:

In [ ]:
for method in ["hierarchical", "contiguous"]:
    r = tsam.aggregate(
        data, n_clusters=6, period_duration="1D", cluster=ClusterConfig(method=method)
    )
    order = [int(c) for c in r.cluster_assignments]
    runs = sum(1 for i in range(len(order) - 1) if order[i] != order[i + 1]) + 1
    first_seen = list(dict.fromkeys(order))
    print(f"{method}: {runs} runs over {len(order)} days")
    print(f"  clusters by first appearance: {first_seen}")
    print(f"  ...sorted? {first_seen == sorted(first_seen)}\n")

So this is wrong for **every** method, `contiguous` included:

```python
# WRONG — cluster ids carry no chronology
for cluster_id in sorted(result.cluster_representatives.index.unique()):
    ...  # assumes cluster 0 comes first in time
```

Day *i* is represented by `cluster_assignments[i]`, and nowhere else.

---

* [How long will this take?](runtime.ipynb) — what each method costs to run.
* [Representations](representations.ipynb) — the companion lever.
* [Working with typical periods](working_with_typical_periods.ipynb) — using the assignments.
* [Comparing clustering methods](../tutorials/comparing_clustering_methods.ipynb) — why they
  disagree.